In [3]:
import torch


In [4]:
import torch
print(torch.__version__)  # Should show 2.6.0+cu124 or 2.6.0+cu121
print(torch.version.cuda)  # Should show 12.4 or 12.1
print(torch.cuda.is_available())  # Should be True
print(torch.cuda.device_count())  # Should be > 0
print(torch.cuda.get_device_name(0))  # Should show your GPU name


2.6.0+cu124
12.4
True
4
NVIDIA GeForce RTX 4090


In [3]:
import os

# Set your directory path
chaptered_dir = "merged_normalized_transcripts"  # or the full path like "/mnt/data/chaptered_output"

# Get all .txt files
chaptered_files = [f for f in os.listdir(chaptered_dir) if f.endswith(".txt")]

# Count unique base filenames
unique_files = set(chaptered_files)

# Output
print(f"📁 Total chaptered .txt files: {len(unique_files)}")
print("🔍 Sample files:")
for f in list(unique_files)[:10]:
    print(" -", f)


📁 Total chaptered .txt files: 435
🔍 Sample files:
 - 6HIsAAElA6U.txt
 - 7ic6oOHobWw.txt
 - 6-Q3JQ0gUcs.txt
 - A7tF6ljEtVU.txt
 - 5qUbo8LDQGE.txt
 - 3eXWiQDYfIs.txt
 - 6pDHTJ1hK7E.txt
 - -g969furGik.txt
 - 8wTtGRSN2so.txt
 - ABIKrrktoAI.txt


In [2]:
import os

# 📁 Folder containing refined chapters
REFINED_DIR = "merged_normalized_transcripts"

# 🔍 Get list of .txt files
files = sorted([f for f in os.listdir(REFINED_DIR) if f.endswith(".txt")])
if not files:
    print("🚫 No files found in the refined output folder.")
else:
    first_file = files[0]
    path = os.path.join(REFINED_DIR, first_file)
    print(f"\n📄 Displaying content of: {first_file}\n{'-'*60}")

    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    print(content[:]) # 🔍 Print first 5000 characters



📄 Displaying content of: -33ywOXKh2M.txt
------------------------------------------------------------
[0.00s - 3.64s]: C एक general purpose object oriented programming language ह
[3.64s - 7.80s]: जसक जन सटउसटरप न C language क extension क तरपर बनय थ
[7.80s - 10.48s]: सटउसटरप एक ऐस programming language बनन चत थ
[10.48s - 13.34s]: ज क C क जस basic और user friendly ह
[13.34s - 16.86s]: और उस क सथ-सथ व चत थ क उसम कछ high level features भ ह
[16.86s - 18.36s]: for example classes and objects
[18.36s - 23.62s]: इस लए सटउसटरप न जब इसक बनय थ त इसक initial नम उनह न C with classes दय थ
[23.62s - 25.40s]: आप लग क जनकर क लए बत द क
[25.40s - 29.28s]: 1970 क उस दर म लक programming करन क लए
[29.28s - 33.40s]: low level machine instructions लख करत थ ज क एक भगत जटल कर रह थ
[33.40s - 36.56s]: C म low level C जस features त हत ह ह
[36.56s - 39.62s]: उस क सथ-सथ कछ high level generic features भ हत ह
[39.62s - 42.72s]: ज क इस खस और अपन आब म एक अलग language बनत ह
[42.72s - 48.78s]: इस जररत क समझत हए 1979 म सटउ

In [4]:
import os
import re
import json
import subprocess
from datetime import timedelta
from tqdm import tqdm
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from keybert import KeyBERT

# Ensure spaCy model is installed
try:
    import en_core_web_sm
except ImportError:
    subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"])
    import en_core_web_sm

import spacy
nlp = en_core_web_sm.load()

# Load mBART model
model_name = "facebook/mbart-large-50-many-to-many-mmt"
tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)

# Load KeyBERT
kw_model = KeyBERT()

# Convert seconds to HH:MM:SS
def seconds_to_hhmmss(seconds):
    return str(timedelta(seconds=int(seconds)))

# Parse transcript lines like: [0.00s - 3.64s]: Text here
def parse_transcript(filepath):
    segments = []
    pattern = re.compile(r"\[(\d+\.\d+)s\s*-\s*(\d+\.\d+)s\]:\s*(.*)")

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            match = pattern.match(line.strip())
            if match:
                start_time = float(match.group(1))
                text = match.group(3).strip()
                if text:
                    segments.append((start_time, text))
    return segments

# Chunk segments into pieces of N seconds (default 300s = 5min)
def chunk_segments(segments, chunk_duration=300):
    chunks = []
    current_chunk = []
    start_time = segments[0][0] if segments else 0

    for start_sec, text in segments:
        if start_sec - start_time <= chunk_duration:
            current_chunk.append((start_sec, text))
        else:
            chunks.append((start_time, current_chunk))
            start_time = start_sec
            current_chunk = [(start_sec, text)]

    if current_chunk:
        chunks.append((start_time, current_chunk))
    return chunks

# Summarize a chunk using mBART (Hindi → English)
def summarize_mbart(text, src_lang="hi_IN", tgt_lang="en_XX"):
    tokenizer.src_lang = src_lang
    encoded = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)
    generated_tokens = model.generate(**encoded, forced_bos_token_id=tokenizer.lang_code_to_id[tgt_lang])
    summary = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    return summary

# Generate chapter list from chunks
def generate_chapters(chunks):
    chapter_list = []
    for start_sec, chunk in chunks:
        full_text = " ".join([text for _, text in chunk])
        try:
            doc = nlp(full_text)
            clean_text = " ".join([sent.text.strip() for sent in doc.sents])

            summary = summarize_mbart(clean_text)
            keywords = kw_model.extract_keywords(summary, top_n=1)
            title = keywords[0][0].title() if keywords else summary[:40]

            chapter_list.append({
                "start_time": seconds_to_hhmmss(start_sec),
                "title": title,
                "summary": summary
            })
        except Exception as e:
            print(f"⚠️ Error generating chapter at {seconds_to_hhmmss(start_sec)}: {e}")
    return chapter_list

# Save output to JSON
def save_chapters(chapters, output_path):
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(chapters, f, indent=2, ensure_ascii=False)

# Process all .txt transcripts in folder
def process_folder(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    txt_files = [f for f in os.listdir(input_folder) if f.endswith(".txt")]

    print(f"📂 Found {len(txt_files)} transcript files.")
    for filename in tqdm(txt_files):
        input_path = os.path.join(input_folder, filename)
        output_path = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}_chapters.json")

        try:
            segments = parse_transcript(input_path)
            if not segments:
                print(f"⚠️ Skipping empty or malformed file: {filename}")
                continue

            chunks = chunk_segments(segments)
            chapters = generate_chapters(chunks)
            save_chapters(chapters, output_path)
            print(f"✅ Chapters written to: {output_path}")

        except Exception as e:
            print(f"❌ Error processing {filename}: {e}")

# Entry point
if __name__ == "__main__":
    process_folder("merged_normalized_transcripts", "chapter_outputs_mbart")


📂 Found 435 transcript files.


  0%|          | 1/435 [00:50<6:03:15, 50.22s/it]

✅ Chapters written to: chapter_outputs_mbart/-33ywOXKh2M_chapters.json


  0%|          | 2/435 [06:57<28:29:24, 236.87s/it]

✅ Chapters written to: chapter_outputs_mbart/-7q7hKspV7Y_chapters.json


...

✅ Chapters written to: chapter_outputs_mbart/AOvaOjN9dIQ_chapters.json


100%|█████████▉| 434/435 [22:45:35<03:53, 233.95s/it]

✅ Chapters written to: chapter_outputs_mbart/Apcz-7hPoTw_chapters.json


100%|██████████| 435/435 [22:52:18<00:00, 189.28s/it]

✅ Chapters written to: chapter_outputs_mbart/ApGx3caiNzI_chapters.json


In [11]:
import os

def count_unique_files(folder_path):
    # List all files in the folder
    all_files = os.listdir(folder_path)
    
    # Use a set to count unique file names
    unique_files = set(all_files)
    
    print(f"Total files found: {len(all_files)}")
    print(f"Unique files found: {len(unique_files)}")

# Replace with your actual folder path
folder_path = "chapter_outputs_cleaned"
count_unique_files(folder_path)


Total files found: 172
Unique files found: 172


In [16]:
import os
import json

def print_first_json_file_contents(folder_path):
    # List and sort all .json files
    json_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.json')])
    
    if not json_files:
        print("No JSON files found in the folder.")
        return

    first_file = json_files[220]
    file_path = os.path.join(folder_path, first_file)
    
    print(f"📂 First JSON file: {first_file}\n")
    
    # Open and print the contents
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        print(json.dumps(data, indent=4, ensure_ascii=False))

# Replace with your actual folder path
folder_path = "chapter_outputs_mbart"
print_first_json_file_contents(folder_path)


📂 First JSON file: 4nMwZhF_D-g_chapters.json

[
    {
        "start_time": "0:00:00",
        "title": "Cyprus",
        "summary": "You know in one of my books, Shivrais wife, Puttala Rani asks him, what freedom means to you? [יוनॉल चेंज] So he tells her in simple words, freedom means to me that I must do what I want to do and I must refuse what I do not want to do. Which was not the case in his growing up years? [יוनॉल चेंज] Which was not the case with most of the Maratha clan he knew. He wanted to attack the mores. First he wrote a lot of letters to mores, please join me. So mores said nothing doing, who are you? [יוनॉल चेंज] Mores were also very arrogant, they were coming to Shivrais territory and he is jagir and collecting taxes and you know they were torturing farmers. So Shivrai said stop all this and join me against my fight against Adilchai."
    },
    {
        "start_time": "0:10:05",
        "title": "Jairdars",
        "summary": "Jagirdars were more relaxed. Once the la

In [3]:
import os
import json
import nltk
from nltk.translate.bleu_score import sentence_bleu
from textstat import flesch_reading_ease
import numpy as np 

nltk.download('punkt')
# Only needed once

# Helper function to convert HH:MM:SS to seconds
def timestamp_to_seconds(timestamp):
    h, m, s = map(int, timestamp.split(":"))
    return h * 3600 + m * 60 + s

# Compute BLEU score between title and summary
def compute_bleu(title, summary):
    reference = nltk.word_tokenize(summary.lower())
    candidate = nltk.word_tokenize(title.lower())
    return sentence_bleu([reference], candidate, weights=(0.5, 0.5))

# Readability score of summary
def compute_readability(summary):
    return flesch_reading_ease(summary)

# Time gaps between consecutive chapters
def compute_time_gaps(chapters):
    gaps = []
    for i in range(1, len(chapters)):
        prev = timestamp_to_seconds(chapters[i - 1]['start_time'])
        curr = timestamp_to_seconds(chapters[i]['start_time'])
        gaps.append(curr - prev)
    return gaps

# Infer durations of chapters using the next chapter's start time
def compute_chapter_lengths(chapters):
    lengths = []
    for i in range(len(chapters) - 1):
        start = timestamp_to_seconds(chapters[i]['start_time'])
        end = timestamp_to_seconds(chapters[i + 1]['start_time'])
        lengths.append(end - start)
    return lengths

# Process one JSON file
def process_json_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        chapters = json.load(f)

    bleu_scores = []
    readability_scores = []

    for chapter in chapters:
        title = chapter.get("title", "")
        summary = chapter.get("summary", "")
        if title and summary:
            bleu_scores.append(compute_bleu(title, summary))
            readability_scores.append(compute_readability(summary))

    time_gaps = compute_time_gaps(chapters)
    chapter_lengths = compute_chapter_lengths(chapters)

    return {
        "bleu": np.mean(bleu_scores) if bleu_scores else 0,
        "readability": np.mean(readability_scores) if readability_scores else 0,
        "gap": np.mean(time_gaps) if time_gaps else 0,
        "length": np.mean(chapter_lengths) if chapter_lengths else 0
    }




Error processing -33ywOXKh2M_chapters.json: 
**********************************************************************
  Resource punkt_tab not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')
  
  For more information see: https://www.nltk.org/data.html

  Attempted to load tokenizers/punkt_tab/english/

  Searched in:
    - '/root/nltk_data'
    - '/venv/main/nltk_data'
    - '/venv/main/share/nltk_data'
    - '/venv/main/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************

Error processing -7q7hKspV7Y_chapters.json: 
**********************************************************************
  Resource punkt_tab not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')
  
  For more information see: https://w

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [5]:
import os
import json
from sacrebleu.metrics import BLEU
from textstat import flesch_reading_ease
import numpy as np

# Initialize BLEU scorer
bleu_metric = BLEU(effective_order=True)

# Helper function to convert HH:MM:SS to seconds
def timestamp_to_seconds(timestamp):
    h, m, s = map(int, timestamp.split(":"))
    return h * 3600 + m * 60 + s

# Compute BLEU score between title and summary using SacreBLEU
def compute_bleu(title, summary):
    return bleu_metric.sentence_score(title, [summary]).score

# Compute readability score
def compute_readability(summary):
    return flesch_reading_ease(summary)

# Compute time gaps between chapters
def compute_time_gaps(chapters):
    gaps = []
    for i in range(1, len(chapters)):
        prev = timestamp_to_seconds(chapters[i - 1]['start_time'])
        curr = timestamp_to_seconds(chapters[i]['start_time'])
        gaps.append(curr - prev)
    return gaps

# Compute length of each chapter
def compute_chapter_lengths(chapters):
    lengths = []
    for i in range(len(chapters) - 1):
        start = timestamp_to_seconds(chapters[i]['start_time'])
        end = timestamp_to_seconds(chapters[i + 1]['start_time'])
        lengths.append(end - start)
    return lengths

# Process one JSON file
def process_json_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        chapters = json.load(f)

    bleu_scores = []
    readability_scores = []

    for chapter in chapters:
        title = chapter.get("title", "")
        summary = chapter.get("summary", "")
        if title and summary:
            bleu_scores.append(compute_bleu(title, summary))
            readability_scores.append(compute_readability(summary))

    time_gaps = compute_time_gaps(chapters)
    chapter_lengths = compute_chapter_lengths(chapters)

    return {
        "bleu": np.mean(bleu_scores) if bleu_scores else 0,
        "readability": np.mean(readability_scores) if readability_scores else 0,
        "gap": np.mean(time_gaps) if time_gaps else 0,
        "length": np.mean(chapter_lengths) if chapter_lengths else 0
    }

# Process all JSON files in a folder
def process_all_json_files(folder_path):
    bleu_all, readability_all, gap_all, length_all = [], [], [], []

    for filename in os.listdir(folder_path):
        if filename.endswith(".json"):
            filepath = os.path.join(folder_path, filename)
            try:
                metrics = process_json_file(filepath)
                bleu_all.append(metrics["bleu"])
                readability_all.append(metrics["readability"])
                gap_all.append(metrics["gap"])
                length_all.append(metrics["length"])
            except Exception as e:
                print(f"Error processing {filename}: {e}")

    # Final results
    print("\n📊 Evaluation Results Across All Files:")
    print(f"Average BLEU Score: {np.mean(bleu_all):.4f}")
    print(f"Average Readability Score: {np.mean(readability_all):.2f}")
    print(f"Average Time Gap Between Chapters (sec): {np.mean(gap_all):.2f}")
    print(f"Average Chapter Duration (sec): {np.mean(length_all):.2f}")

# Example usage
if __name__ == "__main__":
    folder_path = "chapter_outputs_mbart"  # Replace with your actual path
    process_all_json_files(folder_path)



📊 Evaluation Results Across All Files:
Average BLEU Score: 1.6402
Average Readability Score: 25.06
Average Time Gap Between Chapters (sec): 284.14
Average Chapter Duration (sec): 284.14
